# Kaggle DFDC Data Download & Setup

This notebook provides an automated pipeline for downloading and extracting the official Kaggle [Deepfake Detection Challenge (DFDC)](https://www.kaggle.com/c/deepfake-detection-challenge) dataset using the Kaggle API.

In [ ]:
# Install Kaggle CLI and required dependencies
!pip install -q kaggle pandas tqdm

In [ ]:
import os
import json
import zipfile
from pathlib import Path
import pandas as pd

# Define dataset storage directories
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
EXTRACTED_DIR = DATA_DIR / "extracted"

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target directory initialized at: {DATA_DIR.resolve()}")

## 1. Verify Kaggle API Authentication

Ensure your `kaggle.json` API token is located at `~/.kaggle/kaggle.json` with correct permissions (`chmod 600 ~/.kaggle/kaggle.json`).

In [ ]:
# Authenticate Kaggle API
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("Kaggle API successfully authenticated!")
except Exception as e:
    print(f"Authentication error: {e}")
    print("Please place kaggle.json in ~/.kaggle/ and try again.")

## 2. Download Official DFDC Competition Data

Download competition files using Kaggle CLI.

In [ ]:
# Download competition dataset
!kaggle competitions download -c deepfake-detection-challenge -p {RAW_DIR}

## 3. Extract Archives & Verify Metadata

In [ ]:
zip_files = list(RAW_DIR.glob("*.zip"))
print(f"Found {len(zip_files)} zip archives to extract.")

for z_path in zip_files:
    print(f"Extracting {z_path.name}...")
    with zipfile.ZipFile(z_path, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_DIR)

print("Extraction complete!")

In [ ]:
# Inspect metadata.json inside extracted folders
metadata_files = list(EXTRACTED_DIR.glob("**/metadata.json"))
print(f"Found {len(metadata_files)} metadata.json files.")

if metadata_files:
    with open(metadata_files[0], 'r') as f:
        meta_data = json.load(f)
    df = pd.DataFrame.from_dict(meta_data, orient='index')
    print(f"Sample folder contains {len(df)} video entries:")
    print(df.head())
    print("\nLabel balance:")
    print(df['label'].value_counts())